# MCDI502 · Taller 1:Gestión de Datos y Tecnologías Big Data con Apache Spark
## Grupo 4 

**Asignatura:** MCDI502 · Gestión de Datos y Tecnologías Big Data
**Programa:** Magíster en Ciencias de Datos e Inteligencia Artificial, Universidad Andrés Bello
**Integrantes:** Felipe Díaz Toro, Pablo Ríos Passteni, Guillermo Subiabre Chacón, Ninoska Yévenes Hernández
**Herramientas:** Apache Spark 3.5.9 (PySpark)
**Datos:** modelo histórico de los Juegos Olímpicos (deportista, equipo, evento, resultado, juego)

## 1. Configuración inicial del entorno
### 1.1 Java y Spark, y variables de entorno

In [1]:
# Java 17 y PySpark 3.5.9 ya están instalados en el entorno de conda del grupo (grupo4_mcdi502)
# Se crean las variables de entorno necesarias: JAVA_HOME apunta al Java de ese entorno
import os
import sys

os.environ["JAVA_HOME"] = os.path.join(sys.prefix, "lib", "jvm")

print("✅ Variables de entorno configuradas")

✅ Variables de entorno configuradas


### 1.2 SparkSession y SparkContext

In [2]:
# Importar findspark para inicializar Spark
import findspark
findspark.init()

# Importar SparkSession (clase principal para trabajar con Spark)
from pyspark.sql import SparkSession

# Crear la sesión de Spark en modo local usando todos los núcleos disponibles
spark = SparkSession.builder \
        .master("local[*]") \
        .appName('Taller1 Grupo4') \
        .getOrCreate()

# Crear el SparkContext (necesario para trabajar con RDDs)
sc = spark.sparkContext

print("✅ Sesión Spark creada")
print("Spark Version:", spark.version)

26/11/06 18:56:59 WARN Utils: Your hostname, MacBook-Pro-de-Ninoska.local resolves to a loopback address: 127.0.0.1; using 192.168.1.117 instead (on interface en0)
26/11/06 18:56:59 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/11/06 18:56:59 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


✅ Sesión Spark creada
Spark Version: 3.5.9


## 2. RDDs: creación y unión (ID 3.1)
### 2.0 Ruta de los datos 


In [3]:
# El notebook está en notebooks/, por eso los datos se buscan un nivel arriba
ruta = '../data/raw/taller1_sumativa2/'

### 2.1 RDD deportista (6 particiones)

In [4]:
# Cada elemento del RDD es una línea del archivo (deportista.csv no tiene encabezado)
deportista = sc.textFile(ruta + 'deportista.csv', 6)

print("Particiones de deportista:", deportista.getNumPartitions())
deportista.take(3)

Particiones de deportista: 6


['1,A Dijiang,1,24,180,80,199',
 '2,A Lamusi,1,23,170,60,199',
 '3,Gunnar Nielsen Aaby,1,24,0,0,273']

### 2.2 RDD deportista2

In [5]:
deportista2 = sc.textFile(ruta + 'deportista2.csv')

print("Particiones de deportista2:", deportista2.getNumPartitions())
deportista2.take(3)

Particiones de deportista2: 2


['67787,Lee BongJu,1,27,167,56,970',
 '67788,Lee BuTi,1,23,164,54,203',
 '67789,Anthony N. Buddy Lee,1,34,172,62,1096']

### 2.3 Unión de RDDs: deportistaTotal

In [6]:
# union() es una transformación: junta ambos RDDs sin eliminar duplicados
deportistaTotal = deportista.union(deportista2)

print("Particiones de deportistaTotal:", deportistaTotal.getNumPartitions())

Particiones de deportistaTotal: 8


### 2.4 Cantidad de registros de deportistaTotal

In [7]:
# count() es una acción: dispara el cálculo
print("Registros en deportistaTotal:", deportistaTotal.count())

Registros en deportistaTotal: 135571


### 2.5 Conversión a DataFrame deportista

In [8]:
from pyspark.sql.functions import col

columnas = ["deportista_id", "nombre", "genero", "edad", "altura", "peso", "equipo_id"]

# split(",")[:7]: una fila de deportista.csv (id 16587) trae una coma de más y tendría 8 campos
deportista = deportistaTotal.map(lambda linea: linea.split(",")[:7]).toDF(columnas)

# Todo llega como texto: se convierte a tipo numérico (peso a double por sus decimales)
deportista = deportista.withColumn("deportista_id", col("deportista_id").cast("int")) \
                       .withColumn("genero", col("genero").cast("int")) \
                       .withColumn("edad", col("edad").cast("int")) \
                       .withColumn("altura", col("altura").cast("int")) \
                       .withColumn("peso", col("peso").cast("double")) \
                       .withColumn("equipo_id", col("equipo_id").cast("int"))

deportista.show(5)
deportista.printSchema()

+-------------+--------------------+------+----+------+----+---------+
|deportista_id|              nombre|genero|edad|altura|peso|equipo_id|
+-------------+--------------------+------+----+------+----+---------+
|            1|           A Dijiang|     1|  24|   180|80.0|      199|
|            2|            A Lamusi|     1|  23|   170|60.0|      199|
|            3| Gunnar Nielsen Aaby|     1|  24|     0| 0.0|      273|
|            4|Edgar Lindenau Aabye|     1|  34|     0| 0.0|      278|
|            5|Christine Jacoba ...|     2|  21|   185|82.0|      705|
+-------------+--------------------+------+----+------+----+---------+
only showing top 5 rows

root
 |-- deportista_id: integer (nullable = true)
 |-- nombre: string (nullable = true)
 |-- genero: integer (nullable = true)
 |-- edad: integer (nullable = true)
 |-- altura: integer (nullable = true)
 |-- peso: double (nullable = true)
 |-- equipo_id: integer (nullable = true)



## 3. RDDs: transformaciones (ID 3.2)

### 3.1 RDD MayorEdad

In [9]:
# Un deportista es mayor de edad si tiene 18 años o más
# split(",")[:7] protege de la fila corrupta (deportista_id 16587 trae una coma extra)
MayorEdad = deportistaTotal.filter(lambda linea: int(linea.split(",")[:7][3]) >= 18)

print("Deportistas mayores de edad:", MayorEdad.count())
MayorEdad.take(5)

Deportistas mayores de edad: 123244


['1,A Dijiang,1,24,180,80,199',
 '2,A Lamusi,1,23,170,60,199',
 '3,Gunnar Nielsen Aaby,1,24,0,0,273',
 '4,Edgar Lindenau Aabye,1,34,0,0,278',
 '5,Christine Jacoba Aaftink,2,21,185,82,705']

### 3.2 RDD Deportistas_mujer

In [10]:
# El campo genero/sexo: 1 = hombre, 2 = mujer
Deportistas_mujer = deportistaTotal.filter(lambda linea: linea.split(",")[:7][2] == "2")

print("Deportistas mujeres:", Deportistas_mujer.count())
Deportistas_mujer.take(5)

Deportistas mujeres: 33981


['5,Christine Jacoba Aaftink,2,21,185,82,705',
 '8,Cornelia Cor Aalten Strannood ,2,18,168,0,705',
 '13,Minna Maarit Aalto,2,30,159,55.5,350',
 '14,Pirjo Hannele Aalto Mattila ,2,32,171,65,350',
 '21,Ragnhild Margrethe Aamodt,2,27,163,0,742']

### 3.3 deportistaTotal en mayúsculas

In [11]:
# Convierte a mayúsculas cada línea completa del RDD deportistaTotal
deportistaTotalMayus = deportistaTotal.map(lambda linea: linea.upper())
deportistaTotalMayus.take(5)

['1,A DIJIANG,1,24,180,80,199',
 '2,A LAMUSI,1,23,170,60,199',
 '3,GUNNAR NIELSEN AABY,1,24,0,0,273',
 '4,EDGAR LINDENAU AABYE,1,34,0,0,278',
 '5,CHRISTINE JACOBA AAFTINK,2,21,185,82,705']

## 4. DataFrames: creación e integración (ID 3.1)

Esta sección le correspondía a Guillermo; se adelantó para poder aplicar las columnas calculadas de la Sección 7 sobre el DataFrame integrado (como pide la rúbrica), no solo sobre `deportista`. Guillermo puede revisar/ajustar esta parte.

### 4.1 Crear los DataFrames Equipos, Resultado y Juego

In [12]:
from pyspark.sql.functions import broadcast

# Equipos, Resultado y Juego se cargan directo: no tienen problemas de formato
Equipos = spark.read.csv(ruta + 'equipo.csv', sep=',', inferSchema=True, header=True)
Resultado = spark.read.csv(ruta + 'resultados.csv', sep=';', inferSchema=True, header=True)
Juego = spark.read.option("multiline", "true").json(ruta + 'juegos.json')

print("Equipos:", Equipos.count(), "filas")
print("Resultado:", Resultado.count(), "filas")
print("Juego:", Juego.count(), "filas")

Equipos: 1184 filas
Resultado: 271116 filas
Juego: 51 filas


### 4.1.1 Crear el DataFrame Evento (requiere reparar el archivo)

**Hallazgo de calidad de datos:** `evento.csv` tiene un bug de exportación. Cuando el nombre del evento contiene una coma (ej. *"Speed Skating Men's 1,500 metres"*), la fila completa quedó envuelta en una capa extra de comillas (`"30,""Art...Event"",14"`), lo que rompe el parser CSV estándar. Afecta 280 de 764 filas (36.6%) y, si no se repara, se pierden ~64.500 filas al unir con `Resultado` más adelante (lo detectamos comparando el conteo antes/después del join). Se repara quitando esa capa extra de comillas antes de parsear como CSV.

In [13]:
textoEvento = sc.textFile(ruta + 'evento.csv')
headerEvento = textoEvento.first()

def limpiar_doble_comillado(linea):
    # Si la fila completa quedó envuelta en comillas por el bug de exportación,
    # se retira esa capa extra y se des-escapan las comillas dobles internas
    if linea.startswith('"') and linea.endswith('"'):
        return linea[1:-1].replace('""', '"')
    return linea

filasEvento = textoEvento.filter(lambda l: l != headerEvento).map(limpiar_doble_comillado)
Evento = spark.read.csv(filasEvento, sep=',', inferSchema=True, header=False).toDF(*headerEvento.split(','))

print("Evento:", Evento.count(), "filas")
Evento.filter(col("evento").contains(",")).show(5, truncate=False)

Evento: 765 filas
+----------+-------------------------------------------------+----------+
|cod_evento|evento                                           |deporte_id|
+----------+-------------------------------------------------+----------+
|30        |Art Competitions Mixed Sculpturing, Unknown Event|14        |
|38        |Wrestling Men's Light-Heavyweight, Greco-Roman   |18        |
|40        |Speed Skating Men's 1,500 metres                 |5         |
|41        |Gymnastics Men's Team All-Around, Free System    |13        |
|50        |Athletics Men's 1,500 metres                     |7         |
+----------+-------------------------------------------------+----------+
only showing top 5 rows



### 4.2 Optimización de los DataFrames

`Evento`, `Equipos` y `Juego` son tablas de dimensión pequeñas (765, 1184 y 51 filas) que se reutilizan varias veces (acá y en la Sección 8), frente a `Resultado` que tiene 271.116 filas. Se cachean para no releer/parsear los archivos cada vez, y se transmiten (`broadcast`) en los joins de 4.3 en vez de mezclar (shuffle) la tabla grande.

In [14]:
Evento.cache()
Equipos.cache()
Juego.cache()

print("Evento.is_cached:", Evento.is_cached)
print("Equipos.is_cached:", Equipos.is_cached)
print("Juego.is_cached:", Juego.is_cached)

Evento.is_cached: True
Equipos.is_cached: True
Juego.is_cached: True


### 4.3 DataFrame integrado (deportista + evento + equipo + resultado + juego)

`Resultado` es la tabla central (conecta deportista, juego y evento). Se arma paso a paso para evitar columnas ambiguas en los joins.

In [15]:
paso1 = Resultado.join(broadcast(Juego), "juego_id")
paso2 = paso1.join(broadcast(Evento), paso1["evento_id"] == Evento["cod_evento"])
paso3 = paso2.join(deportista, "deportista_id")
integrado = paso3.join(broadcast(Equipos), paso3["equipo_id"] == Equipos["id"])

print("Resultado original:", Resultado.count(), "filas")
print("DataFrame integrado:", integrado.count(), "filas")
print("(la diferencia son resultados con evento_id='#N/A' en el dataset original, no un error de carga)")

integrado.select("deportista_id", "nombre", "medalla", "evento", "equipo", "ano").show(5, truncate=False)

Resultado original: 271116 filas


DataFrame integrado: 270907 filas
(la diferencia son resultados con evento_id='#N/A' en el dataset original, no un error de carga)


+-------------+--------------------+-------+-------------------------------------+------+-------------+
|deportista_id|nombre              |medalla|evento                               |equipo|ano          |
+-------------+--------------------+-------+-------------------------------------+------+-------------+
|148          |Kathryn Katie Abbott|NA     |Sailing Women's Three Person Keelboat|Canada|2008 Verano  |
|463          |Agnese boltia       |NA     |Alpine Skiing Women's Super G        |Latvia|2014 Invierno|
|463          |Agnese boltia       |NA     |Alpine Skiing Women's Giant Slalom   |Latvia|2014 Invierno|
|463          |Agnese boltia       |NA     |Alpine Skiing Women's Slalom         |Latvia|2014 Invierno|
|471          |Ashraf Abou ElHassan|NA     |Volleyball Men's Volleyball          |Egypt |2000 Verano  |
+-------------+--------------------+-------+-------------------------------------+------+-------------+
only showing top 5 rows



## 5. Paralelismo (ID 3.4)

_Pendiente_

## 6. Inspección del DataFrame (ID 3.1)

_Pendiente_

## 7. Columnas calculadas

Se calculan sobre el DataFrame `integrado` de la Sección 4 (no sobre `deportista` solo), tal como pide la rúbrica: *"se observan las columnas en salidas del DataFrame integrado"*.

**Hallazgo de calidad de datos:** el 23.7% de las filas de `integrado` corresponde a deportistas con `altura=0` y/o `peso=0` (dato faltante en el dataset original). El `IMC` se deja en `NULL` para esas filas en vez de inventar un valor — se documenta el porcentaje en la Sección 7.1.

### 7.1 Columna calculada IMC

In [16]:
from pyspark.sql.functions import col, round, when

# El 23.7% de las filas de integrado tiene altura=0 y/o peso=0 en el dataset original (dato
# faltante del censo histórico, no un error de carga). Se mide para dejarlo documentado.
total = integrado.count()
faltantes = integrado.filter((col("altura") == 0) | (col("peso") == 0)).count()
print(f"Filas con altura o peso faltante (=0): {faltantes} de {total} ({faltantes/total:.1%})")

# Se tratan explícitamente como NULL antes de dividir (en vez de depender de que 0/0
# lo haga implícito): así el IMC queda NULL solo cuando el dato de origen falta.
# IMC = peso (kg) / altura^2 (m); la altura del dataset está en cm, se convierte a metros
altura_m = when(col("altura") == 0, None).otherwise(col("altura") / 100)
peso_valido = when(col("peso") == 0, None).otherwise(col("peso"))

integrado = integrado.withColumn("IMC", round(peso_valido / altura_m ** 2, 2))

integrado.select("nombre", "altura", "peso", "IMC").show(5)

Filas con altura o peso faltante (=0): 64196 de 270907 (23.7%)


+--------------------+------+----+-----+
|              nombre|altura|peso|  IMC|
+--------------------+------+----+-----+
|Kathryn Katie Abbott|   164|63.0|23.42|
|       Agnese boltia|   166|60.0|21.77|
|       Agnese boltia|   166|60.0|21.77|
|       Agnese boltia|   166|60.0|21.77|
|Ashraf Abou ElHassan|   186|86.0|24.86|
+--------------------+------+----+-----+
only showing top 5 rows



### 7.2 Columna calculada Descripcion_sexo

In [17]:
from pyspark.sql.functions import when

# sexo=1 es hombre, sexo=2 es mujer
integrado = integrado.withColumn(
    "Descripcion_sexo",
    when(col("genero") == 1, "Hombre").when(col("genero") == 2, "Mujer").otherwise("Desconocido")
)

integrado.select("nombre", "genero", "Descripcion_sexo").show(5)

+--------------------+------+----------------+
|              nombre|genero|Descripcion_sexo|
+--------------------+------+----------------+
|Kathryn Katie Abbott|     2|           Mujer|
|       Agnese boltia|     2|           Mujer|
|       Agnese boltia|     2|           Mujer|
|       Agnese boltia|     2|           Mujer|
|Ashraf Abou ElHassan|     1|          Hombre|
+--------------------+------+----------------+
only showing top 5 rows



## 8. Agregaciones requeridas (ID 3.3)

_Pendiente_